# AI Engineering Level 1 - Module 3: Data Handling & Preprocessing
This notebook contains Lab 1 to Lab 4 implementations for Module 3.

--- 
## Lab 1 - NumPy Vectorization & Broadcasting
**Objective:** Master NumPy's vectorized operations for ML feature matrix computations.

In [ ]:
import numpy as np
np.random.seed(42)

# Feature matrix: 5 samples x 4 features
X = np.random.randn(5, 4)
print('Feature Matrix X shape:', X.shape)
print(X.round(3))

# Vectorized normalization (Z-score)
means = X.mean(axis=0)
stds = X.std(axis=0)
X_norm = (X - means) / stds  # Broadcasting: (5,4) - (4,) / (4,)
print('\nNormalized (mean~0, std~1):')
print(f'Column means: {X_norm.mean(axis=0).round(4)}')
print(f'Column stds: {X_norm.std(axis=0).round(4)}')

# Dot product (forward pass simulation)
weights = np.random.randn(4, 1)
bias = np.zeros(1)
logits = X_norm @ weights + bias
print('\nLogits shape:', logits.shape)
print('Logits:', logits.flatten().round(4))

# Broadcasting outer operation
vec_a = np.array([1, 2, 3])[:, np.newaxis]  # (3,1)
vec_b = np.array([10, 20])  # (2,)
result = vec_a + vec_b  # (3,2)
print('\nBroadcast result (3x2):')
print(result)

--- 
## Lab 2 - Pandas DataFrame Profiling & Cleaning
**Objective:** Profile and clean a realistic tabular dataset with missing values and outliers.

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

# Simulate customer dataset
df = pd.DataFrame({
    'customer_id': range(1, 11),
    'age': [25, np.nan, 35, 28, np.nan, 42, 31, np.nan, 27, 38],
    'income': [50000, 65000, np.nan, 48000, 72000, np.nan, 55000, 61000, np.nan, 80000],
    'segment': ['A', 'B', 'A', np.nan, 'C', 'B', np.nan, 'A', 'C', 'B'],
    'churn': [0, 1, 0, 0, 1, 0, 1, 0, 1, 0]
})
print('Shape:', df.shape)
print('Missing values:\n', df.isnull().sum())

# Imputation
df['age'] = df['age'].fillna(df['age'].median())
df['income'] = df['income'].fillna(df['income'].mean())
df['segment'] = df['segment'].fillna(df['segment'].mode()[0])
print('\nAfter imputation - missing values:\n', df.isnull().sum())

# Descriptive stats
print('\nAge stats:', df['age'].describe().round(2))
print('Churn rate:', df['churn'].mean())

--- 
## Lab 3 - GroupBy Aggregations & SQL-Style Merges
**Objective:** Aggregate multi-dimensional data and join datasets like SQL for feature engineering.

In [ ]:
import pandas as pd

# Sales dataset
sales = pd.DataFrame({
    'region': ['North','South','North','East','South','East','North'],
    'product': ['A','B','A','C','A','B','C'],
    'revenue': [12000, 8500, 15000, 9200, 7800, 11000, 13500],
    'units': [120, 85, 150, 92, 78, 110, 135]
})

# GroupBy aggregation
region_stats = sales.groupby('region').agg(
    total_revenue=('revenue', 'sum'),
    avg_units=('units', 'mean'),
    product_count=('product', 'nunique')
).round(2)
print('Region Statistics:')
print(region_stats)

# Pivot table
pivot = sales.pivot_table(values='revenue', index='region', columns='product', aggfunc='sum', fill_value=0)
print('\nPivot Table (Revenue):')
print(pivot)

# SQL-style merge
region_meta = pd.DataFrame({'region': ['North','South','East'], 'manager': ['Alice','Bob','Carol']})
merged = sales.merge(region_meta, on='region', how='left')
print('\nMerged with manager info:')
print(merged[['region','product','revenue','manager']].head())

--- 
## Lab 4 - Data Export Pipeline
**Objective:** Build a complete data preprocessing and export pipeline.

In [ ]:
import pandas as pd
import numpy as np
from io import StringIO

# Raw CSV simulation
raw_csv = '''id,name,age,salary,department
1,John Smith,32,75000,Engineering
2,Jane Doe,,82000,Marketing
3,Bob Johnson,45,68000,
4,Alice Brown,28,91000,Engineering
5,Charlie Davis,55,,Sales'''

df = pd.read_csv(StringIO(raw_csv))
print('Raw data:')
print(df)

# Cleaning pipeline
df['age'] = df['age'].fillna(df['age'].median())
df['salary'] = df['salary'].fillna(df['salary'].median())
df['department'] = df['department'].fillna('Unknown')
df['name_clean'] = df['name'].str.strip().str.title()
df['salary_band'] = pd.cut(df['salary'], bins=[0, 70000, 85000, float('inf')], labels=['Low', 'Mid', 'High'])

print('\nCleaned & Engineered:')
print(df[['name_clean', 'age', 'salary', 'department', 'salary_band']])
print('\nData types:', df.dtypes.to_dict())